In [28]:
import numpy as np
from netCDF4 import Dataset
import datetime
import tqdm
from regrid import regrid
import xarray as xr
import os
import warnings
warnings.filterwarnings("ignore")

In [29]:
datapath_in = '/Users/carmennab/Dropbox/OI_PolarSnow/data/SnowModelLG/'
datapath_out = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
lons = np.load('/Users/carmennab/Dropbox/alpha_retracker/data/auxiliary/lon_25km_cent.npy')
lats = np.load('/Users/carmennab/Dropbox/alpha_retracker/data/auxiliary/lat_25km_cent.npy')

ease_lons = np.load(f'{datapath_out}lon_cent.npy')
ease_lats = np.load(f'{datapath_out}lat_cent.npy')

### ERA5

In [34]:
for ver in ['MERRA2','ERA5']:
    for var,path in zip(['snod','sden'],['snow_depth','snow_density']):
    
        if ver == 'MERRA2':
            days = np.arange(0,4595)
            years = np.arange(2010,2023)
        elif ver == 'ERA5':
            days = np.arange(0,4230)
            years = np.arange(2010,2022)

        all_dates = [datetime.date(2010,1,1)+datetime.timedelta(days=int(days)) for days in days]

        #import lon, lat & snod values from original SM dataset

        file = f'{datapath_in}SM_{var}_{ver}_01Aug1980-31Jul2018_v01.nc'
        data = Dataset(file,'r')

        lon = data['x']
        lat = data['y']
        snod = data[var][10745:] # 1/1/2010 onwards only
        snod_orig = np.array(snod) ; lon = np.array(lon) ; lat = np.array(lat)
        snod_orig[snod_orig==-9999] = np.nan

        #import lon, lat & snod values from SM extension
        datapath = '/Users/carmennab/Dropbox/OI_PolarSnow/data/SnowModelLG/'
        file = f'{datapath_in}SM_{var}_{ver}_ease_01Aug2018-31Jul2021.nc'
        data = Dataset(file,'r')
        snod_ext = data[var]
        snod_ext = np.array(snod_ext)
        snod_ext[snod_ext==-9999] = np.nan

        # combine SM_orig and SM_ext
        snow = np.concatenate((snod_orig,snod_ext),axis=0)

        ### add 2022 
        if ver == 'MERRA2':
            datapath = '/Users/carmennab/Dropbox/OI_PolarSnow/data/SnowModelLG/'
            file = f'{datapath_in}SM_{var}_{ver}_ease_01Aug2021-31Jul2022.nc'
            data = Dataset(file,'r')
            snod_ext = data[var]
            snod_ext = np.array(snod_ext)
            snod_ext[snod_ext==-9999] = np.nan   
            snow_all = np.concatenate((snow,snod_ext),axis=0)

        elif ver == 'ERA5':
            snow_all = snow

        for year in tqdm.tqdm(years):

            d1 = datetime.date(year, 1, 1)
            d2 = datetime.date(year, 12, 31)
            dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]

            ease_snows = np.full((len(dates),84,103),np.nan)

            for date_count,date in enumerate(dates):
                for all_date_count,all_date in enumerate(all_dates):

                    if all_date == date:
                        ease_snows[date_count] = regrid(snow_all[all_date_count],lons,lats,ease_lons,ease_lats)

            dates = [date.strftime('%Y%m%d') for date in dates]

             ### create NETCDF
            filepath_out = f'{datapath_out}{path}/{ver}/{year}.nc'
            if os.path.exists(filepath_out) == True: 
                ! rm {filepath_out}
            data_vars = {f'{path}':(['t','x','y'],ease_snows)}
            ds = xr.Dataset(data_vars = data_vars,
                            coords={'longitude':(['x','y'],ease_lons),
                                    'latitude':(['x','y'],ease_lats)})
            ds.to_netcdf(filepath_out,'w')


100%|███████████████████████████████████████████| 12/12 [05:28<00:00, 27.34s/it]
